# Atividade 04 — Problemas: Produção com restrições financeiras e Escala de trabalho

**Aluno**: Matheus Sousa Marinho <br> 
**Matrícula**: 202206132

## **Problema 1**: Produção com restrições financeiras

A Electronics fabrica fones de ouvido sem fio e caixas de som Bluetooth (custos e preços na Tabela 1) e deve definir o plano de produção para dezembro de 2025. Em 1º de dezembro, o estoque de matéria-prima comporta até 100 unidades de cada produto; o balanço inicial consta na Tabela 2.

Durante dezembro, todas as vendas são a prazo (recebimento somente em fevereiro). No período, entram \$2.000 de contas a receber antigas; saem \$1.000 de aluguel, \$1.000 de amortização do empréstimo e os custos de mão de obra, pagos à vista.

Em 1º de janeiro chega um novo lote de matéria-prima (\$2.000) com pagamento para fevereiro. Nessa data, a administração exige saldo de caixa ≥ \$4.000 e índice de liquidez corrente ≥ 2. Determine o plano que maximize a contribuição ao lucro (receita futura − custos variáveis).

### Tabela 1 — Custos e preço de venda (dólares)

| Produto | Preço | M. de obra | Mat.-prima |
| :--- | ---: | ---: | ---: |
| Fones s. fio (x) | 100 | 50 | 30 |
| Cx. Bluetooth (y) | 90 | 35 | 40 |

### Tabela 2 — Balanço em 01/dez/2025 (dólares)

| Ativo circulante | Valor | Passivo circulante | Valor |
| :--- | ---: | :--- | ---: |
| Caixa | 10.000 | Empréstimo | 10.000 |
| Contas a receber | 3.000 | — | — |
| Estoque | 7.000 | — | — |
| **Total** | **20.000** | **Total** | **10.000** |

*Liquidez corrente inicial:* 20.000 / 10.000 = 2.


## Configurando o ambiente

Alinhado ao padrão utilizado em exercícios anteriores: `amplpy` + módulo `open` (solver CBC).

In [3]:
# 1. Instalação
%pip install -U -q amplpy
import sys, subprocess

subprocess.check_call([sys.executable, "-m", "amplpy.modules", "install", "--upgrade", "open"])

Note: you may need to restart the kernel to use updated packages.
$ /home/m9t/Documents/ufg/o-research/.venv/bin/python -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_open --upgrade
Looking in indexes: https://pypi.ampl.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 4.8 MB/s  0:00:09m0:00:01:00:01m
  Attempting uninstall: ampl_module_open
    Found existing installation: ampl-module-open 20251121
    Uninstalling ampl-module-open-20251121:
      Successfully uninstalled ampl-module-open-20251121
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_open.


0

In [5]:
# 2. PATH do AMPL e importação
import os, sys, subprocess

caminho_mod = subprocess.check_output([sys.executable, "-m", "amplpy.modules", "path"]).decode().strip()
os.environ["PATH"] += os.pathsep + caminho_mod

from amplpy import AMPL

print("AMPL pronto (solver padrão configurado por modelo abaixo).")

AMPL pronto (solver padrão configurado por modelo abaixo).


## **Parte I — Formulação do problema**

### Variáveis de decisão:
x = quantidade de fones sem fio produzidos em dezembro
y = quantidade de caixas de som Bluetooth produzidas em dezembro

## **Parte II - Função objetivo**
Contribuição unitária ao lucro:

Fones sem fio: 100 - 50 - 30 = 20
Caixas de som Bluetooth: 90 - 35 - 40 = 15
Z = 20x + 15y -> queremos maximizar esta função

## **Parte III - Restrições**

### Restrições de demanda:
x <= 100, y <= 100

### Restrições de estoque:
50x + 35y <= 6000

### Restrições de liquidez:
20x + 15y >= 2000

### Não negatividade:
x >= 0, y >= 0


In [12]:
def modelo1():
    """Solução do problema 1 via AMPL (com restrição de liquidez).

    Variáveis:
        x = unidades de fones sem fio
        y = unidades de caixas Bluetooth
    """
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x >= 0;
        var y >= 0;

        maximize contribuicao: 20*x + 15*y;

        subject to
        mp_x:     x <= 100;
        mp_y:     y <= 100;
        caixa:    50*x + 35*y <= 6000;
        liquidez: 20*x + 15*y >= 2000;
        """
    )
    ampl.solve()
    ampl.eval("display x, y;")
    ampl.eval("display contribuicao;")
    return ampl


ampl_prob1 = modelo1()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 2500
0 simplex iterations
x = 50
y = 100

contribuicao = 2500



In [14]:
def modelo1_sem_liquidez():
    """Solução do problema 1 via AMPL (sem restrição de liquidez).

    Variáveis:
        x = unidades de fones sem fio
        y = unidades de caixas Bluetooth
    """
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x >= 0;
        var y >= 0;

        maximize contribuicao: 20*x + 15*y;

        subject to
        mp_x:     x <= 100;
        mp_y:     y <= 100;
        caixa:    50*x + 35*y <= 6000;
        """
    )
    ampl.solve()
    ampl.eval("display x, y;")
    ampl.eval("display contribuicao;")
    return ampl


ampl_prob1 = modelo1_sem_liquidez()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 2500
0 simplex iterations

"option abs_boundtol 1.4210854715202004e-14;"
or "option rel_boundtol 1.4210854715202004e-16;"
will change deduced dual values.

x = 50
y = 100

contribuicao = 2500



## **Questões**

1. **Por que Z = 20x + 15y é contribuição e não lucro contábil de dezembro?**
   R: As vendas de dezembro são a prazo e os custos fixos como o aluguel não dependem de x, y. O lucro Z mede a margem de contribuição  o que a decisão de produção influencia.

2. **Quais restrições estão ativas no ótimo (x=50, y=100, z=2500)?**
   R: `mp_y` (y = 100) e `caixa` (50x + 35y = 6000). `mp_x` e `liquidez` ficam folgadas. Restrição ativa = recurso escasso que limita o lucro; afrouxá-la aumentaria z.

3. **A liquidez foi determinante? Compare com e sem ela.**
   R: Não. z* = 2500 > 2000 → liquidez folgada; com ou sem ela, o ótimo é o mesmo.

4. **E se o caixa mínimo subir a $5.000? E $8.000?**
   R: **$5.000** (`50x + 35y <= 5000`): novo ótimo (30, 100), z = 2100. **$8.000** (`50x + 35y <= 2000`): inviável, pois o máximo de 20x + 15y nessa região é ~857 < 2000 exigido pela liquidez.

5. **E se 30% das vendas de dezembro fossem à vista?**
   R: Entra em caixa `0,30·(100x + 90y) = 30x + 27y`. A restrição de caixa vira `20x + 8y ≤ 6000` (mais folgada). A liquidez **não muda**: o que entra em caixa sai de contas a receber, e AC/PC totais ficam iguais.

## **Problema 2**: Escala de trabalho

Uma empresa de atendimento técnico precisa programar sua equipe para cobrir a demanda **semanal**. Cada operador trabalha **5 dias consecutivos** e descansa **2**. Deve-se decidir **quantos operadores iniciam o ciclo de 5 dias em cada dia da semana**, de forma a **atender a demanda mínima diária** e **minimizar o número total de funcionários**.

### **Tabela 3 — Demanda mínima diária (operadores)**

| Dia | Operadores |
| :--- | ---: |
| Segunda | 17 |
| Terça | 13 |
| Quarta | 15 |
| Quinta | 19 |
| Sexta | 14 |
| Sábado | 16 |
| Domingo | 11 |

### **Parte I — Formulação do problema**

**Variáveis de decisão:**
x_seg, x_ter, x_qua, x_qui, x_sex, x_sab, x_dom, sendo x_i o número de operadores que iniciam no dia x_i.

**Função objetivo:**
min z = x_seg + x_ter + x_qua + x_qui + x_sex + x_sab + x_dom

### **Tabela 4 — Turnos que cobrem cada dia**

| Dia | Turnos de início que cobrem |
| :--- | :--- |
| Seg | Seg, Qui, Sex, Sab, Dom |
| Ter | Seg, Ter, Sex, Sab, Dom |
| Qua | Seg, Ter, Qua, Sab, Dom |
| Qui | Seg, Ter, Qua, Qui, Dom |
| Sex | Seg, Ter, Qua, Qui, Sex |
| Sab | Ter, Qua, Qui, Sex, Sab |
| Dom | Qua, Qui, Sex, Sab, Dom |

**Restrições de cobertura:**

$$
\begin{aligned}
x_{\mathrm{Seg}} + x_{\mathrm{Qui}} + x_{\mathrm{Sex}} + x_{\mathrm{Sab}} + x_{\mathrm{Dom}} &\ge 17 \\
x_{\mathrm{Seg}} + x_{\mathrm{Ter}} + x_{\mathrm{Sex}} + x_{\mathrm{Sab}} + x_{\mathrm{Dom}} &\ge 13 \\
x_{\mathrm{Seg}} + x_{\mathrm{Ter}} + x_{\mathrm{Qua}} + x_{\mathrm{Sab}} + x_{\mathrm{Dom}} &\ge 15 \\
x_{\mathrm{Seg}} + x_{\mathrm{Ter}} + x_{\mathrm{Qua}} + x_{\mathrm{Qui}} + x_{\mathrm{Dom}} &\ge 19 \\
x_{\mathrm{Seg}} + x_{\mathrm{Ter}} + x_{\mathrm{Qua}} + x_{\mathrm{Qui}} + x_{\mathrm{Sex}} &\ge 14 \\
x_{\mathrm{Ter}} + x_{\mathrm{Qua}} + x_{\mathrm{Qui}} + x_{\mathrm{Sex}} + x_{\mathrm{Sab}} &\ge 16 \\
x_{\mathrm{Qua}} + x_{\mathrm{Qui}} + x_{\mathrm{Sex}} + x_{\mathrm{Sab}} + x_{\mathrm{Dom}} &\ge 11
\end{aligned}
$$

**Não negatividade e integralidade:** $x_i \in \mathbb{Z}_{\ge 0}$ para todo $i \in \{\mathrm{Seg},\ldots,\mathrm{Dom}\}$.




In [15]:
def modelo2():
    """Problema 2 — escala 5×2 (CBC resolve MILP com variáveis inteiras)."""
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """
        var x_seg integer >= 0;
        var x_ter integer >= 0;
        var x_qua integer >= 0;
        var x_qui integer >= 0;
        var x_sex integer >= 0;
        var x_sab integer >= 0;
        var x_dom integer >= 0;

        minimize total_func:
            x_seg + x_ter + x_qua + x_qui + x_sex + x_sab + x_dom;

        subject to
        c_seg: x_seg + x_qui + x_sex + x_sab + x_dom >= 17;
        c_ter: x_ter + x_seg + x_dom + x_sab + x_sex >= 13;
        c_qua: x_qua + x_ter + x_seg + x_dom + x_sab >= 15;
        c_qui: x_qui + x_qua + x_ter + x_seg + x_dom >= 19;
        c_sex: x_sex + x_qui + x_qua + x_ter + x_seg >= 14;
        c_sab: x_sab + x_sex + x_qui + x_qua + x_ter >= 16;
        c_dom: x_dom + x_sab + x_sex + x_qui + x_qua >= 11;
        """
    )
    ampl.solve()
    ampl.eval("display x_seg, x_ter, x_qua, x_qui, x_sex, x_sab, x_dom;")
    ampl.eval("display total_func;")
    return ampl


ampl_prob2 = modelo2()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 23
0 simplex iterations
x_seg = 2
x_ter = 6
x_qua = 0
x_qui = 7
x_sex = 0
x_sab = 3
x_dom = 5

total_func = 23



## **Questões:**

1. **Qual é a solução ótima contínua? Como interpretar frações de operadores?**
   R: Retirando `integer` (relaxação LP), o ótimo é

   $$z_{LP}^{*} = \tfrac{67}{3} \approx 22{,}33$$

   com $(x_{\mathrm{Seg}}, x_{\mathrm{Ter}}, x_{\mathrm{Qua}}, x_{\mathrm{Qui}}, x_{\mathrm{Sex}}, x_{\mathrm{Sab}}, x_{\mathrm{Dom}}) \approx (6{,}33,\ 5,\ 0{,}33,\ 7{,}33,\ 0,\ 3{,}33,\ 0)$.

   As frações não são implementáveis, pois tratando-se de pessoas por exemplo, não tem como contabilizar 1/3 pessoas. Porém são úteis como limitante inferior do MILP: como o ótimo inteiro é $\ge 22{,}33$, e $x_i \in \mathbb{Z}$, conclui-se $z^{*} \ge 23$ valor efetivamente atingido pelo CBC. Alternativamente, pode-se interpretar a fração como um turno parcial ou média ao longo de várias semanas, mas na operação semanal a leitura correta é arredondar via MILP.


2. **Quais dias apresentam folga positiva na solução ótima? O que isso significa para a gestão da equipe?**
   R: Pelo modelo2 apresentado, os dias com folga positiva são quinta, sábado e domingo. Para a equipe significa que eles podem ser fixados em um tamanho mínimo. Com dias de folga > 0, possuiria uma ociosidade estrutural. Se olharmos para uma solução contínua, como haverão outras soluções, poderíamos realocar as folgas para outros dias.
   
3. **Por que as restrições de cobertura têm estrutura cíclica? O que mudaria se a jornada fosse de 6 dias seguidos com 1 de folga?**
   R: A semana é um ciclo de 7 dias que se repete: quem começa em $s$ trabalha $s, s+1, \ldots, s+4 \pmod 7$, então cada $x_s$ aparece em 5 restrições consecutivas módulo 7 (matriz circulante, como no Problema da Escala de Ônibus). Com jornada **6×1**, cada $x_s$ passa a aparecer em **6** restrições ($s, \ldots, s+5 \pmod 7$) e o piso teórico cai de $\lceil 105/5 \rceil = 21$ para $\lceil 105/6 \rceil = 18$ atende-se a mesma demanda com **menos operadores**, em troca de cada um trabalhar um dia a mais.

4. **Como o modelo seria adaptado para minimizar o custo total, sabendo que operadores que iniciam nos fins de semana recebem 20% a mais?**
   R: Restrições inalteradas; muda apenas a função objetivo, atribuindo peso 1,2 a Sáb e Dom:
   
   $$\min\ c = x_{\mathrm{Seg}} + x_{\mathrm{Ter}} + x_{\mathrm{Qua}} + x_{\mathrm{Qui}} + x_{\mathrm{Sex}} + 1{,}2\,x_{\mathrm{Sab}} + 1{,}2\,x_{\mathrm{Dom}}$$
   
   O solver passa a evitar iniciar turnos no fim de semana sempre que a cobertura continuar atendida.


---

**Referências:** Goldbarg & Luna (2006); [AMPL](https://ampl.com); [HiGHS](https://highs.dev) / CBC via `amplpy` open module.